In [35]:
%pwd


'/Users/rajasekhark/Downloads/CareFusion-AI/CareFusion-AI'

In [36]:
import os 
os.chdir("../")
%pwd

'/Users/rajasekhark/Downloads/CareFusion-AI'

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Extract text from all PDF files inside a folder
def load_pdf_files(folder_path):
    loader = DirectoryLoader(
        folder_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

# Load all PDFs inside the "Data" folder
extracted_data = load_pdf_files("/Users/rajasekhark/Downloads/CareFusion-AI/CareFusion-AI/Data")

len(extracted_data)


4662

In [4]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Convert a list of Document objects into a minimal form:
    - Keep only page_content
    - Keep only 'source' in metadata
    """
    return [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")}
        )
        for doc in docs
    ]

minimal_docs = filter_to_minimal_docs(extracted_data)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

def text_split(
    docs: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 20
) -> List[Document]:
    """
    Split documents into smaller chunks using RecursiveCharacterTextSplitter.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    return splitter.split_documents(docs)

texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")


Number of chunks: 31532


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

def download_embeddings(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device}
    )

embedding = download_embeddings()


In [8]:
vector = embedding.embed_query("Hello world")
vector

[-0.03447723388671875,
 0.031023329123854637,
 0.006735051982104778,
 0.026109037920832634,
 -0.03936200961470604,
 -0.16030250489711761,
 0.06692399829626083,
 -0.006441477220505476,
 -0.047450486570596695,
 0.014758816920220852,
 0.07087532430887222,
 0.05552753806114197,
 0.01919332891702652,
 -0.02625134028494358,
 -0.010109633207321167,
 -0.026940561830997467,
 0.022307511419057846,
 -0.022226708009839058,
 -0.14969263970851898,
 -0.01749306730926037,
 0.0076763383112847805,
 0.0543522909283638,
 0.0032544590067118406,
 0.03172585740685463,
 -0.08462142944335938,
 -0.029406115412712097,
 0.05159566178917885,
 0.048124078661203384,
 -0.003314804285764694,
 -0.058279234915971756,
 0.04196935519576073,
 0.022210679948329926,
 0.12818878889083862,
 -0.022338949143886566,
 -0.01165628619492054,
 0.06292831897735596,
 -0.03287627547979355,
 -0.09122607111930847,
 -0.031175393611192703,
 0.052699536085128784,
 0.04703482612967491,
 -0.08420304954051971,
 -0.030056189745664597,
 -0.020744

In [9]:
len(vector)

384

In [10]:
# Load environment variables
from dotenv import load_dotenv
import os
from pinecone import Pinecone

load_dotenv()  # loads .env file

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Safety check
if not PINECONE_API_KEY:
    raise ValueError("❌ PINECONE_API_KEY is missing. Check your .env file.")

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY is missing. Check your .env file.")

# Set environment variables for downstream libraries
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

pc

In [11]:
from pinecone import ServerlessSpec

index_name = "carefusion"

# Create index only if it does NOT already exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # matches MiniLM-L6-v2
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists.")

# Connect to the index
index = pc.Index(index_name)
index

Index already exists.


In [12]:
print(pc.list_indexes().names())

['carefusion']


In [13]:
from langchain_pinecone import PineconeVectorStore

index_name = "carefusion"

# Load vector store only once
if "docsearch" not in globals():
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )
    print("VectorStore loaded and ready.")
else:
    print("VectorStore already loaded. Skipping reload.")


VectorStore loaded and ready.


In [14]:
index = pc.Index("carefusion")

In [15]:
index.describe_index_stats()


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 31532}},
 'total_vector_count': 31532,
 'vector_type': 'dense'}

In [16]:
from tqdm.auto import tqdm

index = pc.Index("carefusion")

# Check existing vector count
stats = index.describe_index_stats()
existing_count = stats.get("total_vector_count", 0)

expected_count = len(texts_chunk)

print(f"Existing vectors: {existing_count}")
print(f"Expected vectors: {expected_count}")

# If already upserted → skip
if existing_count >= expected_count:
    print("✔️ Vectors already upserted. Skipping upsert.")
else:
    print("⏳ Upserting missing vectors...")

    batch_size = 200
    vectors = []

    for i, chunk in enumerate(tqdm(texts_chunk, desc="Upserting to Pinecone")):
        # Skip if this ID already exists
        vector_id = f"chunk-{i}"
        if i < existing_count:
            continue

        vec = embedding.embed_query(chunk.page_content)

        vectors.append({
            "id": vector_id,
            "values": vec,
            "metadata": {
                "text": chunk.page_content,
                **chunk.metadata
            }
        })

        if len(vectors) == batch_size:
            index.upsert(vectors)
            vectors = []

    if vectors:
        index.upsert(vectors)

    print("✔️ Upsert complete.")


Existing vectors: 31532
Expected vectors: 31532
✔️ Vectors already upserted. Skipping upsert.


In [17]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [18]:
def show_retrieved(docs, preview_chars=600, show_full=False):
    """
    Nicely display retrieved document chunks with clean formatting.
    
    Args:
        docs: List of Document objects
        preview_chars: Number of characters to preview
        show_full: If True, prints full chunk text
    """
    if not docs:
        print("⚠️ No documents retrieved.")
        return

    for i, doc in enumerate(docs, start=1):
        print(f"\n{'='*25} Retrieved Chunk {i} {'='*25}")

        text = doc.page_content or ""
        meta = doc.metadata or {}

        # Safe preview
        if show_full:
            display_text = text
        else:
            display_text = text[:preview_chars] + ("..." if len(text) > preview_chars else "")

        print("\n📄 **Content Preview:**\n")
        print(display_text)

        print("\n🧩 **Metadata:**")
        for k, v in meta.items():
            print(f"  - {k}: {v}")

        print("\n" + "="*70)


In [34]:
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for m in models.data:
    print(m.id)


canopylabs/orpheus-arabic-saudi
allam-2-7b
llama-3.1-8b-instant
openai/gpt-oss-120b
moonshotai/kimi-k2-instruct-0905
llama-3.3-70b-versatile
groq/compound
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
moonshotai/kimi-k2-instruct
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-20b
canopylabs/orpheus-v1-english
qwen/qwen3-32b
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b


In [36]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

print("Groq LLM loaded successfully.")


Groq LLM loaded successfully.


In [38]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Format retrieved docs safely ---
def format_docs(docs, max_chars=4000):
    if not docs:
        return "No relevant context found."
    text = "\n\n".join(d.page_content for d in docs)
    return text[:max_chars]

# --- Prompt ---
prompt = ChatPromptTemplate.from_template("""
You are a helpful medical assistant. Use ONLY the retrieved context to answer.

Question:
{question}

Context:
{context}

Answer clearly, concisely, and avoid hallucinating.
""")

# --- RAG Chain ---
rag_chain = (
    {
        "question": lambda x: x["question"],
        "context": (
            RunnableLambda(lambda x: retriever.invoke(x["question"]))
            | RunnableLambda(format_docs)
        ),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Updated RAG chain ready.")

Updated RAG chain ready.


In [39]:
def ask_bot(query):
    return rag_chain.invoke({"question": query})


In [41]:
print(ask_bot("What causes acne?"))


The exact cause of acne is unknown, but it occurs when hair follicles become clogged by shed skin cells and sebum. Several risk factors have been identified, including hormonal changes, age, gender, disease, and heredity.
